In [3]:
# -*- coding: utf-8 -*-
"""
Created on Tue Apr 21 17:00:17 2026

@author: Darko Panic
"""

# -*- coding: utf-8 -*-
"""
CRISP-DM Notebook: Stack Overflow Developer Survey (2025)
Business Question 1 (BQ1): Career & Compensation

- Countries: USA and Germany
- Population: Full-time employed only
- Year: 2025-only
- Cut: two-sided trimming on target (q_low, q_high)
- Models: DummyRegressor, RidgeCV, HistGradientBoostingRegressor (sklearn)
- NEW: Feature-Ranking via NORMALIZED SHAP Group Importance (instead of Pearson corr ranking)
- SHAP plots: aggregated back to original features

scikit-learn version: 1.6.1
"""

# ============================================================
# (0) Imports & Settings
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor

# SHAP (optional but required for the requested ranking + explanation plots)
try:
    import shap
    SHAP_AVAILABLE = True
except Exception as _e:
    SHAP_AVAILABLE = False
    print("SHAP is not available. Install via: pip install shap")
    print("Error:", _e)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

plt.rcParams["figure.figsize"] = (10, 5)
sns.set_context("notebook")



In [4]:
# ============================================================
# Helper functions
# ============================================================

def eval_regression(y_true, y_pred, label=""):
    """Evaluate regression on the given targets (log-space here)."""
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    print(f"\n--- {label} ---")
    print(f"R²   : {r2:.3f}")
    print(f"MAE  : {mae:.3f}")
    print(f"RMSE : {rmse:.3f}")
    return {"r2": r2, "mae": mae, "rmse": rmse}


def clean_yearscode_inplace(df, col="YearsCode"):
    """Convert YearsCode to numeric (handle special strings)."""
    if col not in df.columns:
        return
    df[col] = df[col].replace({"Less than 1 year": 0.5, "More than 50 years": 50})
    df[col] = pd.to_numeric(df[col], errors="coerce")


def clean_workexp_inplace(df, col="WorkExp"):
    """Convert WorkExp to numeric."""
    if col not in df.columns:
        return
    df[col] = pd.to_numeric(df[col], errors="coerce")


def make_preprocessor(X, dense_output_for_hgbr):
    """ColumnTransformer: numeric imputer + categorical imputer + OHE."""
    numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat_cols = [c for c in X.columns if c not in numeric_cols]

    num_tf = Pipeline([("imputer", SimpleImputer(strategy="median"))])

    # For HGBR we use dense output
    if dense_output_for_hgbr:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        sparse_threshold = 0.0
    else:
        ohe = OneHotEncoder(handle_unknown="ignore")
        sparse_threshold = 0.3

    cat_tf = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", ohe)
    ])

    return ColumnTransformer(
        transformers=[("num", num_tf, numeric_cols), ("cat", cat_tf, cat_cols)],
        remainder="drop",
        sparse_threshold=sparse_threshold
    )


# --- (ALT) Robust correlation ranking (kept as fallback only) ---
def onehot_corr_feature_ranking(df, target_col, candidate_features):
    """
    Rank candidate feature names by max abs correlation after one-hot encoding.
    Robust mapping back to the ORIGINAL feature names (no broken prefixes like 'SO', 'JobSatPoints', etc.).
    """
    work = df[candidate_features + [target_col]].copy()
    work[target_col] = pd.to_numeric(work[target_col], errors="coerce")
    work = work.dropna(subset=[target_col])

    X = pd.get_dummies(work[candidate_features], dummy_na=False)
    y = work[target_col]

    corr = X.apply(lambda c: c.corr(y)).sort_values(ascending=False)

    # Match one-hot column names back to original feature via longest prefix match
    cand_sorted = sorted(candidate_features, key=len, reverse=True)
    base_scores = {}

    for col_name, val in corr.items():
        matched = None
        for feat in cand_sorted:
            if col_name == feat or col_name.startswith(feat + "_"):
                matched = feat
                break
        if matched is None:
            continue
        base_scores[matched] = max(base_scores.get(matched, 0), abs(val))

    return pd.Series(base_scores).sort_values(ascending=False)


# ============================================================
# NEW: Normalized SHAP Feature Ranking (Group importance)
# ============================================================

def onehot_shap_feature_ranking_normalized(
    df,
    target_col,
    candidate_features,
    pipeline_for_ranking,
    background_size=200,
    explain_size=1000,
    random_state=42,
):
    """
    SHAP-äquivalentes, NORMALISIERTES Feature-Ranking.

    normImportance(feature) =
        sum_j mean(|SHAP_j|) / sqrt(n_onehot(feature))

    -> fair für:
       - numerische Features
       - Features mit vielen One-Hot-Kategorien
    """
    if not SHAP_AVAILABLE:
        raise RuntimeError("SHAP is not available, cannot compute SHAP ranking.")

    work = df[candidate_features + [target_col]].copy()
    work[target_col] = pd.to_numeric(work[target_col], errors="coerce")
    work = work.dropna(subset=[target_col])

    if len(work) == 0:
        return pd.Series(dtype=float)

    X = work[candidate_features]

    # Subsampling
    n_bg = min(background_size, len(X))
    n_ex = min(explain_size, len(X))

    X_bg = X.sample(n=n_bg, random_state=random_state)
    X_ex = X.sample(n=n_ex, random_state=random_state)

    pre = pipeline_for_ranking.named_steps["pre"]
    model = pipeline_for_ranking.named_steps["model"]

    X_bg_t = pre.transform(X_bg)
    X_ex_t = pre.transform(X_ex)

    # Names after preprocessing (incl. OHE)
    try:
        feat_names = pre.get_feature_names_out(X.columns)
    except Exception:
        feat_names = [f"f_{i}" for i in range(X_ex_t.shape[1])]

    feat_names = [str(f) for f in feat_names]

    # SHAP
    explainer = shap.Explainer(model, X_bg_t)
    shap_vals = explainer(X_ex_t).values

    if shap_vals.ndim == 3:
        shap_vals = shap_vals[:, :, 0]

    mean_abs = np.abs(shap_vals).mean(axis=0)

    # Map transformed -> original by longest prefix match
    cand_sorted = sorted(candidate_features, key=len, reverse=True)

    sum_abs = {}
    counts = {}

    for j, fname in enumerate(feat_names):
        # sklearn ColumnTransformer prefixes like "num__WorkExp" or "cat__DevType_..."
        fname_clean = fname.split("__", 1)[-1]
        matched = None
        for feat in cand_sorted:
            if fname_clean == feat or fname_clean.startswith(feat + "_"):
                matched = feat
                break
        if matched is None:
            continue

        sum_abs[matched] = sum_abs.get(matched, 0.0) + float(mean_abs[j])
        counts[matched] = counts.get(matched, 0) + 1

    norm_scores = {f: (sum_abs[f] / np.sqrt(max(counts[f], 1))) for f in sum_abs}
    return pd.Series(norm_scores).sort_values(ascending=False)


def median_mode_scenario(df, cols):
    """Create a single-row scenario using median for numeric and mode for categorical columns."""
    row = {}
    for c in cols:
        if pd.api.types.is_numeric_dtype(df[c]):
            row[c] = df[c].median()
        else:
            row[c] = df[c].mode().iloc[0]
    return pd.DataFrame([row])


def apply_comp_cut_two_sided_quantile(df, target_col, q_low=0.01, q_high=0.99):
    """Remove rows below q_low quantile and above q_high quantile of the target."""
    low_val = df[target_col].quantile(q_low)
    high_val = df[target_col].quantile(q_high)
    df_f = df[(df[target_col] >= low_val) & (df[target_col] <= high_val)].copy()
    removed = len(df) - len(df_f)
    removed_share = removed / max(len(df), 1)
    return df_f, low_val, high_val, removed, removed_share


def is_employed_fulltime_only(series: pd.Series) -> pd.Series:
    """
    Include full-time employed respondents.
    Treat 'Employed' as full-time.
    Exclude part-time and self-employment.
    """
    is_employed = series.str.contains("Employed", case=False, na=False)
    is_part_time = series.str.contains("part-time", case=False, na=False)
    is_self_employed = series.str.contains(
        "Independent contractor|freelancer|self-employed|Not employed|Student|Retired|I prefer not to say",
        case=False,
        na=False
    )
    return is_employed & ~is_part_time & ~is_self_employed


# ============================================================
# Predicted vs Actual plot (back-transformed to original unit)
# ============================================================

def _sanitize_label(text: str) -> str:
    """Make a safe filename label."""
    text = text.strip()
    text = re.sub(r"[^\w\-_\. ]", "_", text)
    return text.replace(" ", "_")


def plot_pred_vs_actual_from_log(
    y_true_log,
    y_pred_log,
    run_label: str,
    output_dir: str = "bq1_pred_vs_actual",
    max_points: int = 12000,
    random_state: int = 42,
    use_log_axes: bool = True
):
    """Predicted vs Actual in original units; inputs in log1p-space."""
    os.makedirs(output_dir, exist_ok=True)

    y_true = np.expm1(np.asarray(y_true_log, dtype=float))
    y_pred = np.expm1(np.asarray(y_pred_log, dtype=float))
    y_true = np.clip(y_true, 0, None)
    y_pred = np.clip(y_pred, 0, None)

    n = len(y_true)
    if n > max_points:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(n, size=max_points, replace=False)
        y_true_plot = y_true[idx]
        y_pred_plot = y_pred[idx]
    else:
        y_true_plot = y_true
        y_pred_plot = y_pred

    r2_orig = r2_score(y_true, y_pred)
    rmse_orig = root_mean_squared_error(y_true, y_pred)

    plt.figure(figsize=(7.5, 6.5))
    plt.scatter(y_true_plot, y_pred_plot, s=10, alpha=0.25, edgecolor="none")
    plt.grid(True, alpha=0.25)

    xy_min = max(1.0, float(np.min(np.concatenate([y_true_plot, y_pred_plot]))))
    xy_max = float(np.max(np.concatenate([y_true_plot, y_pred_plot])))
    plt.plot([xy_min, xy_max], [xy_min, xy_max], linestyle="--", linewidth=2)

    if use_log_axes:
        plt.xscale("log")
        plt.yscale("log")

    plt.xlabel("Actual ConvertedCompYearly (back-transformed)")
    plt.ylabel("Predicted ConvertedCompYearly (back-transformed)")
    plt.title(f"Predicted vs Actual – {run_label}")

    text = f"R² (orig)  = {r2_orig:.3f}\nRMSE (orig)= {rmse_orig:,.0f}\nN test     = {n:,}"
    plt.text(
        0.03, 0.97, text,
        transform=plt.gca().transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
    )

    plt.tight_layout()
    out_path = os.path.join(output_dir, f"pred_vs_actual_{_sanitize_label(run_label)}.png")
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()

    return {"r2_orig": r2_orig, "rmse_orig": rmse_orig, "plot_path": out_path}


# ============================================================
# SHAP utilities (Pipeline-safe + aggregated back to original features)
# ============================================================

def _get_feature_names_out_safe(preprocessor):
    """Get output feature names from ColumnTransformer."""
    if hasattr(preprocessor, "get_feature_names_out"):
        return np.array(preprocessor.get_feature_names_out())
    return None


def _map_transformed_to_original(trans_feature_names, original_features):
    """
    Map transformed feature names back to original feature names.
    Example: 'cat__DevType_Developer, back-end' -> 'DevType'
    """
    original_features = list(original_features)
    mapping = []
    # Use longest prefix match for safety (handles underscores/spaces)
    orig_sorted = sorted(original_features, key=len, reverse=True)

    for tf in trans_feature_names:
        tf_clean = tf.split("__", 1)[1] if "__" in tf else tf
        matched = None
        for feat in orig_sorted:
            if tf_clean == feat or tf_clean.startswith(feat + "_"):
                matched = feat
                break
        mapping.append(matched)
    return mapping


def _make_numeric_for_shap_plot(X_df: pd.DataFrame) -> pd.DataFrame:
    """Convert object columns to category codes for SHAP beeswarm coloring."""
    X_plot = X_df.copy()
    for c in X_plot.columns:
        if X_plot[c].dtype == "object" or pd.api.types.is_categorical_dtype(X_plot[c]):
            X_plot[c] = pd.Categorical(X_plot[c]).codes
    return X_plot


def plot_shap_for_pipeline_aggregated(
    pipeline: Pipeline,
    X_eval_raw: pd.DataFrame,
    original_features: list,
    run_label: str,
    output_dir: str = "bq1_shap_plots",
    max_samples: int = 2000,
    random_state: int = 42
):
    """
    Compute SHAP values in transformed feature space and aggregate contributions
    back to the original feature list (summing one-hot components).
    Saves:
      - beeswarm summary plot
      - bar plot (mean abs SHAP)
    Returns file paths.
    """
    if not SHAP_AVAILABLE:
        return None, None

    os.makedirs(output_dir, exist_ok=True)

    present_features = [f for f in original_features if f in X_eval_raw.columns]
    X_eval = X_eval_raw[present_features].copy()

    if len(X_eval) > max_samples:
        X_eval = X_eval.sample(n=max_samples, random_state=random_state)

    preprocessor = pipeline.named_steps["pre"]
    model = pipeline.named_steps["model"]

    X_trans = preprocessor.transform(X_eval)

    trans_feature_names = _get_feature_names_out_safe(preprocessor)
    if trans_feature_names is None:
        trans_feature_names = np.array([f"f_{i}" for i in range(X_trans.shape[1])])

    # Background for SHAP
    bg_size = min(200, X_trans.shape[0])
    rng = np.random.default_rng(random_state)
    bg_idx = rng.choice(X_trans.shape[0], size=bg_size, replace=False)
    X_bg = X_trans[bg_idx]

    # Explainer
    explainer = shap.Explainer(model, X_bg)
    shap_exp = explainer(X_trans)

    shap_vals = shap_exp.values
    if shap_vals.ndim != 2:
        shap_vals = shap_vals[..., 0]

    # Aggregate SHAP values back to original features
    mapping = _map_transformed_to_original(trans_feature_names, present_features)
    agg_vals = np.zeros((shap_vals.shape[0], len(present_features)), dtype=float)
    feat_to_idx = {f: i for i, f in enumerate(present_features)}

    for j, orig_feat in enumerate(mapping):
        if orig_feat is None:
            continue
        agg_vals[:, feat_to_idx[orig_feat]] += shap_vals[:, j]

    # Prepare numeric feature values for beeswarm coloring
    X_plot = _make_numeric_for_shap_plot(X_eval[present_features])

    safe_label = _sanitize_label(run_label)

    # Beeswarm plot
    plt.figure()
    shap.summary_plot(
        agg_vals,
        features=X_plot,
        feature_names=present_features,
        show=False
    )
    plt.title(f"BQ1 SHAP Summary (aggregated) – {run_label}")
    plt.tight_layout()
    summary_path = os.path.join(output_dir, f"bq1_shap_summary_{safe_label}.png")
    plt.savefig(summary_path, dpi=200, bbox_inches="tight")
    plt.close()

    # Bar plot (mean abs SHAP)
    mean_abs = np.mean(np.abs(agg_vals), axis=0)
    order = np.argsort(mean_abs)[::-1]
    plt.figure(figsize=(7, 4))
    plt.bar(np.array(present_features)[order], mean_abs[order])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Mean |SHAP value|")
    plt.title(f"BQ1 Global Feature Importance (SHAP) – {run_label}")
    plt.tight_layout()
    bar_path = os.path.join(output_dir, f"bq1_shap_bar_{safe_label}.png")
    plt.savefig(bar_path, dpi=200, bbox_inches="tight")
    plt.close()

    return summary_path, bar_path

In [27]:
# ============================================================
# (1) GATHER — Load & split data (2025 only)
# ============================================================

FILE_2025 = "survey_results_public_2025.csv"

USA = "United States of America"
GER = "Germany"

df_2025 = pd.read_csv(FILE_2025, low_memory=False)
df_2025["dtYear"] = 2025  # new column to mark the data with year 2025

public_df = pd.concat([df_2025], ignore_index=True)
public_df = public_df[public_df["Country"].isin([USA, GER])]

df_usa_2025 = public_df[(public_df["Country"] == USA) & (public_df["dtYear"] == 2025)]
df_ger_2025 = public_df[(public_df["Country"] == GER) & (public_df["dtYear"] == 2025)]

# Use of function is_employed_fulltime_only to filter
df_usa_2025_emp = df_usa_2025[is_employed_fulltime_only(df_usa_2025["Employment"])]
df_ger_2025_emp = df_ger_2025[is_employed_fulltime_only(df_ger_2025["Employment"])]



In [44]:
print("Number of rows employed: ", (df_usa_2025_emp["Employment"] != "employed").sum())
print(df_usa_2025_emp["Employment"].info())


Number of rows employed:  5413
<class 'pandas.core.series.Series'>
Index: 5413 entries, 21 to 49157
Series name: Employment
Non-Null Count  Dtype 
--------------  ----- 
5413 non-null   object
dtypes: object(1)
memory usage: 84.6+ KB

 None
